In [2]:
import sys
import logging
from typing import Dict, Any, List, Optional, TypedDict
from enum import Enum

# 配置日志渲染
logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("AgentRuntime")

# ============================================================================
# 1. 错误分类与契约定义 (Error Taxonomy & Tool Contracts)
# ============================================================================

class ErrorType(str, Enum):
    RUNTIME_TRANSIENT = "RUNTIME_TRANSIENT"       # 临时网络故障，触发 Retry
    INPUT_VALIDATION_ERROR = "INPUT_VALIDATION"   # 输入参数格式错误，触发 Agent Repair
    AMBIGUITY_ERROR = "AMBIGUITY_ERROR"           # 多实体歧义，触发 Clarification
    NOT_FOUND = "NOT_FOUND"                       # 资源不存在，不可修复

class ToolError(Exception):
    def __init__(self, error_type: ErrorType, message: str, payload: Optional[Dict[str, Any]] = None):
        super().__init__(message)
        self.error_type = error_type
        self.message = message
        self.payload = payload or {}

# 模拟数据库环境
MOCK_EMPLOYEE_DB = [
    {"emp_id": "EMP_1001", "name": "Kevin Zhang", "department": "IT", "leave_balance": 12},
    {"emp_id": "EMP_2038", "name": "Kevin Wang", "department": "Sales", "leave_balance": 8},
    {"emp_id": "EMP_3005", "name": "Alice Chen", "department": "HR", "leave_balance": 15},
]

# ============================================================================
# 2. 底层工具实现 (Tools)
# ============================================================================

def find_employee_by_name(name: str) -> Dict[str, Any]:
    """通过姓名查找员工列表。"""
    logger.info(f"⚙️ [Tool Call] find_employee_by_name(name='{name}')")
    matches = [emp for emp in MOCK_EMPLOYEE_DB if name.lower() in emp["name"].lower()]
    
    if len(matches) == 0:
        raise ToolError(ErrorType.NOT_FOUND, f"未找到名字包含 '{name}' 的员工。")
    elif len(matches) > 1:
        # 抛出歧义错误，附带候选人 payload
        raise ToolError(
            ErrorType.AMBIGUITY_ERROR,
            f"找到多位匹配名字 '{name}' 的员工，需要澄清。",
            payload={"candidates": matches}
        )
    return {"status": "SUCCESS", "employee": matches[0]}

def get_leave_balance(employee_id: str) -> Dict[str, Any]:
    """通过准确的员工 ID (例如 EMP_1001) 查询带薪年假余额。"""
    logger.info(f"⚙️ [Tool Call] get_leave_balance(employee_id='{employee_id}')")
    
    # 强校验契约：格式必须为 EMP_xxxx
    if not employee_id.startswith("EMP_"):
        raise ToolError(
            ErrorType.INPUT_VALIDATION_ERROR,
            f"参数不合规: '{employee_id}' 不是有效的员工 ID 格式！合法 ID 必须以 'EMP_' 开头（例如 EMP_1001）。"
        )
    
    for emp in MOCK_EMPLOYEE_DB:
        if emp["emp_id"] == employee_id:
            return {
                "status": "SUCCESS",
                "emp_id": emp["emp_id"],
                "name": emp["name"],
                "leave_balance": emp["leave_balance"]
            }
            
    raise ToolError(ErrorType.NOT_FOUND, f"未找到员工 ID 为 '{employee_id}' 的记录。")

# ============================================================================
# 3. Agent 状态定义与硬约束 (State & Hard Budgets)
# ============================================================================

class AgentStatus(str, Enum):
    RUNNING = "RUNNING"
    SUCCESS = "SUCCESS"
    NEED_CLARIFICATION = "NEED_CLARIFICATION"
    FAILED_BUDGET_EXHAUSTED = "FAILED_BUDGET_EXHAUSTED"
    FAILED_UNRECOVERABLE = "FAILED_UNRECOVERABLE"

class AgentState(TypedDict):
    query: str
    messages: List[Dict[str, Any]]
    step_count: int
    repair_count: int
    status: AgentStatus
    final_output: Optional[str]

MAX_STEPS = 6
MAX_REPAIRS = 2

# ============================================================================
# 4. Agent 决策推理引擎 (Deterministic Engine & State Machine)
# ============================================================================

def _handle_tool_error(e: ToolError, state: AgentState):
    """统一记录工具异常日志到消息栈。"""
    logger.warning(f"⚠️ [Tool Error Caught] Type: {e.error_type.value} | Details: {e.message}")
    state["messages"].append({
        "role": "tool_error",
        "error_type": e.error_type,
        "content": e.message,
        "payload": e.payload
    })

def agent_reasoning_and_dispatch(state: AgentState) -> AgentState:
    """
    模拟 LLM 的思考与决策过程。
    Runtime 负责强行判定步数与 Repair 预算，LLM 只负责在预算内思考操作。
    """
    state["step_count"] += 1
    logger.info(f"🤖 [Agent Decision Loop] Step {state['step_count']}/{MAX_STEPS} | Repairs: {state['repair_count']}/{MAX_REPAIRS}")

    # 1. 硬边界检测：Max Steps
    if state["step_count"] > MAX_STEPS:
        state["status"] = AgentStatus.FAILED_BUDGET_EXHAUSTED
        state["final_output"] = "系统失败：超过最大推理步数上限 (MAX_STEPS exhausted)。"
        return state

    last_message = state["messages"][-1]
    
    # --- 场景 A: 初始入口或修复后的尝试 ---
    if last_message["role"] == "user":
        target = "Kevin" if "Kevin" in state["query"] else "InvalidParam"
        try:
            res = get_leave_balance(employee_id=target)
            state["status"] = AgentStatus.SUCCESS
            state["final_output"] = f"员工 {res['name']} ({res['emp_id']}) 剩余年假余额为 {res['leave_balance']} 天。"
        except ToolError as e:
            _handle_tool_error(e, state)

    elif last_message["role"] == "tool_error":
        error_type = last_message["error_type"]
        
        # 2. 检查 Repair 预算
        if state["repair_count"] >= MAX_REPAIRS:
            logger.error("🛑 [Budget Alert] Repair 预算已耗尽，终止 Agent 无休止修复尝试。")
            state["status"] = AgentStatus.FAILED_BUDGET_EXHAUSTED
            state["final_output"] = "系统失败：Agent 自主修补尝试次数超限 (MAX_REPAIRS exhausted)。"
            return state

        if error_type == ErrorType.INPUT_VALIDATION_ERROR:
            # 触发 Agent Repair: 重新规划，先通过名字查找 ID
            state["repair_count"] += 1
            logger.warning(f"🔄 [Agent Repair Loop #{state['repair_count']}] 捕获参数格式错误，重新规划路径：调用 find_employee_by_name()")
            
            search_name = "Kevin" if "Kevin" in state["query"] else "BadUser"
            try:
                find_res = find_employee_by_name(name=search_name)
                emp_id = find_res["employee"]["emp_id"]
                
                # 获取到有效 ID，二次调用目标工具
                leave_res = get_leave_balance(employee_id=emp_id)
                state["status"] = AgentStatus.SUCCESS
                state["final_output"] = f"查询成功！员工 {leave_res['name']} ({leave_res['emp_id']}) 剩余年假余额为 {leave_res['leave_balance']} 天。"
            except ToolError as e:
                _handle_tool_error(e, state)

        elif error_type == ErrorType.AMBIGUITY_ERROR:
            # 遇到歧义，不能猜！将决定权交还给用户 (Information Authority)
            logger.info(" Escalating to Human: 识别到多实体歧义，停止自主循环。")
            candidates = last_message["payload"].get("candidates", [])
            cand_str = " / ".join([f"{c['name']} ({c['department']})" for c in candidates])
            state["status"] = AgentStatus.NEED_CLARIFICATION
            state["final_output"] = f"请问您具体指的是哪一位员工？系统中找到多个匹配项：[{cand_str}]"
            
        else:
            state["status"] = AgentStatus.FAILED_UNRECOVERABLE
            state["final_output"] = f"无法修复的错误: {last_message['content']}"

    return state

# ============================================================================
# 5. 验证三个关键业务场景 (Verification Scenarios)
# ============================================================================

def run_agent_test(query_text: str, override_db=None):
    global MOCK_EMPLOYEE_DB
    if override_db is not None:
        MOCK_EMPLOYEE_DB = override_db
        
    print(f"\n==================================================================")
    print(f"🧪 [Scenario Input] Query: '{query_text}'")
    print(f"==================================================================")
    
    state: AgentState = {
        "query": query_text,
        "messages": [{"role": "user", "content": query_text}],
        "step_count": 0,
        "repair_count": 0,
        "status": AgentStatus.RUNNING,
        "final_output": None
    }
    
    while state["status"] == AgentStatus.RUNNING:
        state = agent_reasoning_and_dispatch(state)
        
    print(f"\n📌 [Final Agent State]")
    print(f" - Status        : {state['status'].value}")
    print(f" - Total Steps   : {state['step_count']}")
    print(f" - Repairs Used  : {state['repair_count']}")
    print(f" - Final Output  : {state['final_output']}\n")

if __name__ == "__main__":
    # 场景 ①: 唯一匹配与成功 Repair
    db_single_kevin = [
        {"emp_id": "EMP_1001", "name": "Kevin Zhang", "department": "IT", "leave_balance": 12},
        {"emp_id": "EMP_3005", "name": "Alice Chen", "department": "HR", "leave_balance": 15},
    ]
    run_agent_test("帮我查 Kevin 还有多少年假", override_db=db_single_kevin)

    # 场景 ②: 两个候选人，无法消歧 -> Clarification Escalation
    db_multi_kevin = [
        {"emp_id": "EMP_1001", "name": "Kevin Zhang", "department": "IT", "leave_balance": 12},
        {"emp_id": "EMP_2038", "name": "Kevin Wang", "department": "Sales", "leave_balance": 8},
    ]
    run_agent_test("帮我查 Kevin 还有多少年假", override_db=db_multi_kevin)

    # 场景 ③: Agent 不断输入非法参数 -> Repair Budget Exhausted
    print(f"\n==================================================================")
    print(f"🧪 [Scenario Input] Budget Exhaustion Test (Endless Error Loop)")
    print(f"==================================================================")
    
    bad_state: AgentState = {
        "query": "查询 InvalidUser 的年假",
        "messages": [{"role": "user", "content": "查询 InvalidUser"}],
        "step_count": 0,
        "repair_count": 0,
        "status": AgentStatus.RUNNING,
        "final_output": None
    }
    
    while bad_state["status"] == AgentStatus.RUNNING:
        bad_state["step_count"] += 1
        logger.info(f"🤖 [Agent Decision Loop] Step {bad_state['step_count']}/{MAX_STEPS} | Repairs: {bad_state['repair_count']}/{MAX_REPAIRS}")
        
        if bad_state["step_count"] > MAX_STEPS:
            bad_state["status"] = AgentStatus.FAILED_BUDGET_EXHAUSTED
            bad_state["final_output"] = "系统失败：超过最大推理步数上限。"
            break
            
        try:
            get_leave_balance(employee_id="InvalidFormat_XYZ")
        except ToolError as e:
            _handle_tool_error(e, bad_state)
            
            if bad_state["repair_count"] >= MAX_REPAIRS:
                logger.error("🛑 [Budget Alert] Repair 预算已耗尽，硬中断生效！")
                bad_state["status"] = AgentStatus.FAILED_BUDGET_EXHAUSTED
                bad_state["final_output"] = "系统失败：Agent 自主修补尝试次数超限 (MAX_REPAIRS exhausted)。"
                break
                
            bad_state["repair_count"] += 1

    print(f"\n📌 [Final Agent State]")
    print(f" - Status        : {bad_state['status'].value}")
    print(f" - Total Steps   : {bad_state['step_count']}")
    print(f" - Repairs Used  : {bad_state['repair_count']}")
    print(f" - Final Output  : {bad_state['final_output']}\n")

2026-09-11 16:03:43,074 - [INFO] - 🤖 [Agent Decision Loop] Step 1/6 | Repairs: 0/2
2026-09-11 16:03:43,075 - [INFO] - ⚙️ [Tool Call] get_leave_balance(employee_id='Kevin')
2026-09-11 16:03:43,077 - [WARNING] - ⚠️ [Tool Error Caught] Type: INPUT_VALIDATION | Details: 参数不合规: 'Kevin' 不是有效的员工 ID 格式！合法 ID 必须以 'EMP_' 开头（例如 EMP_1001）。
2026-09-11 16:03:43,078 - [INFO] - 🤖 [Agent Decision Loop] Step 2/6 | Repairs: 0/2
2026-09-11 16:03:43,079 - [WARNING] - 🔄 [Agent Repair Loop #1] 捕获参数格式错误，重新规划路径：调用 find_employee_by_name()
2026-09-11 16:03:43,080 - [INFO] - ⚙️ [Tool Call] find_employee_by_name(name='Kevin')
2026-09-11 16:03:43,081 - [INFO] - ⚙️ [Tool Call] get_leave_balance(employee_id='EMP_1001')
2026-09-11 16:03:43,083 - [INFO] - 🤖 [Agent Decision Loop] Step 1/6 | Repairs: 0/2
2026-09-11 16:03:43,084 - [INFO] - ⚙️ [Tool Call] get_leave_balance(employee_id='Kevin')
2026-09-11 16:03:43,085 - [WARNING] - ⚠️ [Tool Error Caught] Type: INPUT_VALIDATION | Details: 参数不合规: 'Kevin' 不是有效的员工 ID 格式！合法 ID 必


🧪 [Scenario Input] Query: '帮我查 Kevin 还有多少年假'

📌 [Final Agent State]
 - Status        : SUCCESS
 - Total Steps   : 2
 - Repairs Used  : 1
 - Final Output  : 查询成功！员工 Kevin Zhang (EMP_1001) 剩余年假余额为 12 天。


🧪 [Scenario Input] Query: '帮我查 Kevin 还有多少年假'

📌 [Final Agent State]
 - Status        : NEED_CLARIFICATION
 - Total Steps   : 3
 - Repairs Used  : 1
 - Final Output  : 请问您具体指的是哪一位员工？系统中找到多个匹配项：[Kevin Zhang (IT) / Kevin Wang (Sales)]


🧪 [Scenario Input] Budget Exhaustion Test (Endless Error Loop)

📌 [Final Agent State]
 - Status        : FAILED_BUDGET_EXHAUSTED
 - Total Steps   : 3
 - Repairs Used  : 2
 - Final Output  : 系统失败：Agent 自主修补尝试次数超限 (MAX_REPAIRS exhausted)。

